## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. Just press ▶ on the cell below and wait
for the green **✅ Setup complete**, then run the rest top to bottom.

When it asks to **connect Google Drive**, click **Connect** — that lets the data
file download **only once** (it's saved to your Drive and reused by every
notebook) and saves your figures for your poster. You *can* skip it, but then each
notebook re-downloads the ~470 MB data and your figures won't be saved.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

# Connect Drive so the data is downloaded ONCE (saved to your Drive) and your
# figures persist. If you skip it, we fall back to temporary storage.
print("3/3  connecting Google Drive ...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = "/content/drive/MyDrive/DecodingBrain_data"
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    saved = True
except Exception:
    data_dir = "data"                     # temporary (re-downloads each session)
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    saved = False

os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "synapse_preprocessed.pkl")
os.environ["CAMP_DATA_PATH"] = data_path

if os.path.exists(data_path):
    print("     data already saved in your Drive — skipping download \u26a1")
else:
    print("     downloading the data (~470 MB, one time only) ...")
    import gdown
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY", output=data_path, quiet=False)

print("\n\u2705 Setup complete.",
      "Data + figures are saved in your Drive (DecodingBrain_*)." if saved
      else "Heads up: you skipped Drive, so the data re-downloads each session.")


# Week 3 · Tier 3 — Building a Classifier (Simple → Smart)

**Tier 3 "EEG Data Scientist"** is the machine-learning track, and this is
**Research Goal 1**, the headline question of the study:

> Given a new person's brain signals, can we **predict** whether they have sound
> sensitivity?

We'll train a **logistic regression** classifier and test it honestly with
**Leave-One-Out Cross-Validation (LOOCV)**. But here's the real lesson: your
**first** model will barely beat a coin flip. By the end you'll **engineer better
features** and climb toward the SYNAPSE study's **AUC of ~0.79–0.84** — and,
crucially, understand *exactly what made the difference*.

### By the end of this notebook you will be able to
1. Turn features into an X/y matrix and run leakage-safe LOOCV
2. Diagnose *why* a model underperforms (too many noisy features, small n)
3. **Engineer** better features (the onset time window) and measure the gain
4. Compare simple vs. study-grade feature sets and explain the jump

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import roc_auc_score, confusion_matrix
import camp_utils as cu

data = cu.load_camp_data(verbose=False)
features = pd.read_csv(cu.save_path("features_table.csv"))   # your Week-2 table
print("Loaded Week-2 features:", features.shape)

## 1. Build X and y
Machine-learning models eat two things:
- **X**: a matrix of features (rows = subjects, columns = features)
- **y**: the label for each subject (1 = EXP, 0 = CTRL)

We start with the feature table you built in Week 2.

In [ ]:
feature_cols = [c for c in features.columns if c not in ("subject", "group")]
X = features[feature_cols].fillna(features[feature_cols].mean())
y = (features["group"] == "EXP").astype(int).values

print("X shape:", X.shape, " (subjects × features)")
print("y:", y, "->", y.sum(), "EXP and", (y == 0).sum(), "CTRL")

## 2. Why we can't test on the training data
If a model memorizes all 28 people and then "tests" on those same people, it can
score perfectly by **cheating** — memorizing, not learning. That tells us nothing
about a *new* person.

The fix: **always test on data the model never saw during training.** With only 28
subjects, the best version is **Leave-One-Out**: train on 27, predict the 1 left
out, repeat 28 times so every person gets a fair, held-out prediction.

## 3. One LOOCV fold, by hand
Let's demystify it with a single fold.

In [ ]:
loo = LeaveOneOut()
train_idx, test_idx = next(loo.split(X))
print("Train on", len(train_idx), "subjects; test on subject index", test_idx[0])

# IMPORTANT: fit the scaler on TRAIN ONLY, then apply to the test subject.
scaler = StandardScaler().fit(X.iloc[train_idx])
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X.iloc[train_idx]), y[train_idx])
prob = model.predict_proba(scaler.transform(X.iloc[test_idx]))[0, 1]
print(f"Left-out subject is truly {'EXP' if y[test_idx][0] else 'CTRL'}; "
      f"model predicted EXP-probability = {prob:.2f}")

⚠️ **The leakage rule.** We fit the scaler on *training* subjects only. If you
scale using *all* the data first, information about the test subject leaks into
training and your scores come out falsely high. Keep **all** preprocessing inside
the fold. (We'll see how badly leakage fools you in Notebook 14.)

## 4. The full LOOCV loop
### ✏️ Your turn #1 — complete the loop
Fill in the scaling, fitting, and prediction (mirroring section 3).

In [ ]:
true_labels, pred_probs = [], []
for train_idx, test_idx in LeaveOneOut().split(X):
    # TODO (a): fit a StandardScaler on X.iloc[train_idx]; transform train & test
    scaler = None
    X_train = None   # scaler.transform(X.iloc[train_idx])
    X_test = None    # scaler.transform(X.iloc[test_idx])

    # TODO (b): fit LogisticRegression(max_iter=1000) on X_train, y[train_idx]
    model = None

    # TODO (c): predicted EXP-probability for the test subject
    prob = None      # model.predict_proba(X_test)[0, 1]

    true_labels.append(y[test_idx][0])
    pred_probs.append(prob)

true_labels, pred_probs = np.array(true_labels), np.array(pred_probs)
cu.check(None not in list(pred_probs) and len(pred_probs) == len(y),
         "LOOCV produced a held-out prediction for every subject.",
         "Fill in the scaler, model.fit, and predict_proba lines (copy section 3).")

## 5. Score your first model
**AUC** (Area Under the ROC Curve) measures how well the predicted probabilities
separate EXP from CTRL: 0.5 = coin flip, 1.0 = perfect.

In [ ]:
auc_naive = roc_auc_score(true_labels, pred_probs)
print(f"Model 1 — 'throw in all {X.shape[1]} Week-2 features'")
print(f"   AUC = {auc_naive:.3f}   (0.5 = chance)")
print("\n😐 That's barely better than a coin flip. What went wrong?")

## 6. Diagnosis: too many noisy features for 28 people
We fed **~21 features** into a model trained on only **27 people** per fold. Most
of those features (HLT bands, ERP latencies…) carry little group signal — they're
**noise**, and the model overfits to them. This is the **curse of dimensionality**:
with few subjects, *more features usually hurts*.

Let's test that idea — use just **two** LET features instead of all 21.

In [ ]:
# cu.build_psd_feature_matrix turns (task, period, band) specs into X, y
two_specs = [("let", "full_stim", "gamma"), ("let", "full_stim", "beta")]
X2, y2, names2 = cu.build_psd_feature_matrix(data, two_specs)
auc_two = cu.loocv_auc(X2, y2)   # same leakage-safe LOOCV, packaged as a helper
print("Model 2 — just 2 LET features", names2)
print(f"   AUC = {auc_two:.3f}   (better already — and we used FEWER features!)")

**Whoa.** Fewer features did *better*. So the answer isn't "more data columns" —
it's the **right** columns. Which raises the question: which features actually
carry the group difference?

## 7. Feature engineering: the onset window 🔑
Here's the big idea the Week-2 table missed. We measured power over the **whole**
2-second sound (`full_stim`). But the "central gain" theory says sound-sensitive
brains **over-react to the sudden onset** of a sound. That burst lives in the
first **half second** — the `early_stim` window (0–0.5 s) — and averaging over the
whole 2 s **washes it out**.

The SYNAPSE study's best simple model uses exactly **5 features**, three of them
`early_stim`. They're stored in `cu.PAPER_PSD5`:

In [ ]:
for spec in cu.PAPER_PSD5:
    print("  ", cu.feature_label(spec))

X5, y5, names5 = cu.build_psd_feature_matrix(data, cu.PAPER_PSD5)
auc_paper = cu.loocv_auc(X5, y5)
print(f"\nModel 3 — the study's 5 features (with onset windows)")
print(f"   AUC = {auc_paper:.3f}   (study benchmark ≈ 0.84 🎯)")

**That's the jump.** Same simplified tools, same 28 people, same plain LOOCV — but
by measuring the **right rhythms in the right time window**, the AUC leaps from
chance to study-grade. *Better features, not more features.*

## 8. Tell the story in one figure
Compare all three models side by side.

In [ ]:
labels = ["All 21\nWeek-2 features", "2 LET\nfeatures", "Study's 5\n(onset windows)"]
aucs = [auc_naive, auc_two, auc_paper]
colors = ["#999999", cu.CTRL_COLOR, cu.EXP_COLOR]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, aucs, color=colors)
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8, label="chance")
ax.axhspan(0.79, 0.84, color="green", alpha=0.12, label="study benchmark")
for b, a in zip(bars, aucs):
    ax.text(b.get_x() + b.get_width()/2, a + 0.01, f"{a:.2f}", ha="center", fontweight="bold")
ax.set_ylabel("AUC (held-out)")
ax.set_ylim(0, 1)
ax.set_title("What changed the model: feature choice, not quantity")
ax.legend()
plt.tight_layout()
plt.savefig(cu.save_path("tier3_model_ladder.png"), dpi=300, bbox_inches="tight")
plt.show()

## 9. Report the best model honestly
Beyond AUC, clinicians care about **sensitivity** (do we catch the people who have
the condition?) and **specificity** (do we avoid false alarms?).

In [ ]:
auc_best, true_best, prob_best = cu.loocv_auc(X5, y5, return_predictions=True)
pred_best = (prob_best >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(true_best, pred_best).ravel()
print(f"Best model (study's 5 features):")
print(f"   AUC         = {auc_best:.3f}")
print(f"   Sensitivity = {tp/(tp+fn):.2f}  (caught {tp}/{tp+fn} EXP)")
print(f"   Specificity = {tn/(tn+fp):.2f}  (cleared {tn}/{tn+fp} CTRL)")

# save predictions for Notebook 14 (permutation test + ROC)
pd.DataFrame({"true": true_best, "prob": prob_best}).to_csv(
    cu.save_path("loocv_predictions.csv"), index=False)
print("\nSaved best-model predictions for Notebook 14.")

### ✏️ Your turn #2 — engineer your own feature set
Can you find a feature set that does well? Build your own list of
`(task, period, band)` specs and test it with `cu.loocv_auc`. Ideas:
- swap in **`mid_stim`** or **`late_stim`** windows — does the onset really matter?
- add an **AST** onset feature (aversive-sound onset)
- try **theta** or **alpha** instead of beta/gamma

*Hint:* the periods are `early_stim, mid_stim, late_stim, full_stim, poststim`;
bands are `delta, theta, alpha, beta, gamma`.

In [ ]:
my_specs = [
    ("let", "early_stim", "gamma"),
    # TODO: add 2–4 more (task, period, band) tuples and experiment
]
Xmine, ymine, names_mine = cu.build_psd_feature_matrix(data, my_specs)
my_auc = cu.loocv_auc(Xmine, ymine)
print("Your feature set:", names_mine)
print(f"   AUC = {my_auc:.3f}")
cu.check(len(my_specs) >= 3,
         "Nice — you ran your own feature-engineering experiment!",
         "Add at least 3 features to my_specs and rerun.")

## 🎯 Wrap-up (Tier 3)
You didn't just train a model — you **debugged** one. The lesson that separates
data scientists from button-pushers: **with a small sample, careful feature
choice beats throwing everything in.** The onset window, motivated by the central-
gain theory, is what unlocked study-grade performance.

**But two questions remain:** (1) could a "good" AUC happen by **luck** with 28
people? (2) Did picking those 5 features secretly **peek at the answer**? Notebook
14 — permutation testing and the leakage trap — settles both.